# Trainable Parameter Counts

This notebook counts trainable PyTorch parameters for the fixed factory architectures (`arch1`-`arch7`) and for every `arch8` configuration used in `nas/dnn-output/test-best-runs-150`.

The count is computed with:

```python
sum(p.numel() for p in model.parameters() if p.requires_grad)
```

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import pandas as pd
import yaml


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "packages" / "f110_planning" / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing packages/f110_planning/src")


REPO_ROOT = find_repo_root()
F110_PLANNING_SRC = REPO_ROOT / "packages" / "f110_planning" / "src"
if str(F110_PLANNING_SRC) not in sys.path:
    sys.path.insert(0, str(F110_PLANNING_SRC))

from f110_planning.utils.nn_models import get_architecture


def count_parameters(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


TRAINED_MAP_LABELS = {
    "017f7c": "Yas Marina",
    "08e72b": "Moscow Raceway",
    "193215": "Melbourne",
    "1b391f": "Sakhir",
    "3cd867": "Budapest",
    "3d2630": "Oschersleben",
    "4ec05d": "IMS",
    "60de23": "Sao Paulo",
    "782a1c": "Montreal",
    "a8b20f": "Spielberg",
    "be986b": "Austin",
    "d24a66": "Zandvoort",
    "dc559d": "Catalunya",
    "e6278a": "Hockenheim",
    "f847ab": "Brands Hatch",
    "f926b9": "Sepang",
}

TEST_BEST_RUNS_150 = REPO_ROOT / "nas" / "dnn-output" / "test-best-runs-150"
print("repo root:", REPO_ROOT)
print("arch8 config root:", TEST_BEST_RUNS_150)

## Factory Architectures

These are the fixed architecture factories returned by `get_architecture(arch_id)` for `arch1` through `arch7`.

In [ ]:
factory_rows = []
for arch_id in range(1, 8):
    model = get_architecture(arch_id)
    factory_rows.append(
        {
            "architecture": f"arch{arch_id}",
            "trainable_parameters": count_parameters(model),
        }
    )

factory_param_counts = pd.DataFrame(factory_rows)
display(factory_param_counts.style.format({"trainable_parameters": "{:,}"}))

## A* (`arch8`) Architectures Used

Each training run has three separately sampled target networks: `left_wall_dist`, `track_width`, and `heading_error`. This section counts each target model and then sums the three target counts for the full controller configuration.

In [ ]:
def trial_number_from_name(path: Path) -> int | None:
    match = re.search(r"_trial(\d+)", path.name)
    return int(match.group(1)) if match else None


arch8_rows = []
for cfg_path in sorted(TEST_BEST_RUNS_150.glob("*/*_arch8_trial*.yaml")):
    with cfg_path.open("r", encoding="utf-8") as fh:
        cfg = yaml.safe_load(fh)

    model = get_architecture(8, cfg["model"])
    training_run = cfg_path.parent.name
    dynamic = cfg["model"]["dynamic"]
    conv_layers = dynamic.get("conv_layers", [])

    arch8_rows.append(
        {
            "training_run": training_run,
            "training_track": TRAINED_MAP_LABELS.get(training_run, training_run),
            "target": cfg["data"]["target_col"],
            "trial": trial_number_from_name(cfg_path),
            "trainable_parameters": count_parameters(model),
            "activation": dynamic.get("activation"),
            "n_conv_layers": len(conv_layers),
            "conv_channels": tuple(layer["out_channels"] for layer in conv_layers),
            "pool_sizes": tuple(layer.get("pool_size") for layer in conv_layers),
            "fc_layers": tuple(dynamic.get("fc_layers", [])),
            "config_path": str(cfg_path.relative_to(REPO_ROOT)),
        }
    )

arch8_param_counts = pd.DataFrame(arch8_rows)
display(
    arch8_param_counts.sort_values(["training_track", "target"]).style.format(
        {"trainable_parameters": "{:,}", "trial": "{:.0f}"}
    )
)

In [ ]:
arch8_param_pivot = (
    arch8_param_counts
    .pivot(index="training_track", columns="target", values="trainable_parameters")
    .sort_index()
)
arch8_param_pivot["controller_total"] = arch8_param_pivot.sum(axis=1)

display(arch8_param_pivot.style.format("{:,}"))

In [ ]:
arch8_summary = arch8_param_counts["trainable_parameters"].describe().to_frame("per_target_params")
controller_summary = arch8_param_pivot["controller_total"].describe().to_frame("controller_total_params")

display(arch8_summary.style.format("{:,.1f}"))
display(controller_summary.style.format("{:,.1f}"))